### this notebook is the start of the network wrangler process. 
this starts from scratch, with json and new rail links.





In [1]:
import os
import sys
import pickle

import numpy as np

from pyproj import CRS

from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler import Scenario
from network_wrangler.roadway import write_roadway
from network_wrangler.transit import write_transit

from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_roadway
from met_council_wrangler import metcouncil_transit

from cube_wrangler import Parameters
from cube_wrangler import util
from cube_wrangler import roadway
from cube_wrangler import StandardTransit



Geopandas is not using pyogrio as the I/O engine.                Install pyogrio to benefit from faster I/O.


In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
import logging
logger = logging.getLogger("WranglerLogger")

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    logger.addHandler(handler)

logger.handlers[0].stream = sys.stdout
logger.setLevel(logging.INFO)

# remote i/o

In [4]:
input_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00")
cc_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00")
rail_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")
tran_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")

net_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks")
output_network = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output")

metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler")


In [5]:
project_card_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1")
project_card_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2_v1")
project_card_dir3 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1")
project_card_dir4 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1")

In [6]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


# Load Version00

In [7]:
link_file = os.path.join(input_dir, 'standard_networks', 'links.json')
node_file = os.path.join(input_dir, 'standard_networks', 'nodes.geojson')
shape_file = os.path.join(input_dir, 'standard_networks', 'shapes.geojson')

roadway_net = load_roadway(
    links_file=link_file,
    nodes_file=node_file,
    shapes_file=shape_file,
)

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Read 414131 nodes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\nodes.geojson in 21.41.
Reading links from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\links.json.
Read + transformed 1061566 links from             Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\links.json in 132.18.


In [8]:
roadway_net.links_df.shape

(1061566, 52)

In [9]:
roadway_net.nodes_df.shape

(414131, 12)

In [10]:
transit_net = load_transit(os.path.join(tran_dir,"standard_transit_network"))

Reading GTFS feed tables from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network


\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\transit\io.py:86: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file)


Initializing frequencies
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Initializing routes
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Initializing shapes
Initializing stops
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Initializing trips
Referencing table stop_tim

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)


In [11]:
roadway_net.links_df = roadway_net.links_df[roadway_net.links_df.A != roadway_net.links_df.B]

### Attribute the Network

In [12]:
roadway_net.links_df = roadway_net.links_df.drop('lanes', axis = 1)

In [13]:
# make sure the data types of the boolean columns are correct
roadway_net.links_df['bus_only'] = False

for c in list(set(roadway_net.links_df.columns) & set(metcouncil_parameters.bool_col)):
    roadway_net.links_df[c] = roadway_net.links_df[c].replace(
        {
            np.nan: False,
            "": False,
            "0": False,
            "1": True
        }
    )

In [14]:
r_net = metcouncil_roadway.calculate_number_of_lanes_from_reviewed_network(
    roadway_net=roadway_net,
    parameters=metcouncil_parameters,
)
r_net.links_df.lanes.value_counts()

Calculating Number of Lanes as network variable: 'lanes'
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Calculating Centroid Connector and adding as roadway network variable: centroidconnect
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Finished calculating centroid connector variable: centroidconnect
Finished calculating number of lanes to: lanes


lanes
1.0    1026496
2.0      32020
3.0       2595
4.0        369
5.0         71
6.0          5
7.0          1
Name: count, dtype: int64

In [15]:
r_net.links_df.roadway.value_counts()

roadway
residential       420553
service           264058
footway           117014
tertiary          113519
cycleway           83868
secondary          36639
primary            14067
motorway_link       3604
trunk               3051
motorway            2757
secondary_link       839
trunk_link           601
tertiary_link        538
primary_link         449
Name: count, dtype: int64

In [16]:
r_net = metcouncil_roadway.calculate_assign_group_and_roadway_class_from_reviewed_network(
        roadway_net=r_net,
        parameters=metcouncil_parameters,
)
r_net.links_df.assign_group.value_counts(dropna=False)

Calculating Assignment Group and Roadway Class as network variables: 'assign_group' and 'roadway_class'
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Centroid Connector Variable 'centroidconnect' already in network. Returning without overwriting.
Finished calculating assignment group variable assign_group and roadway class variable roadway_class


assign_group
50.0     294095
101.0    241490
103.0    204010
102.0    117014
7.0      101584
6.0       76802
5.0       15025
15.0       3045
3.0        2590
1.0        2455
4.0        2036
2.0        1258
11.0        153
Name: count, dtype: int64

In [17]:
r_net.links_df.roadway_class.value_counts()

roadway_class
101.0    562514
50.0     304018
40.0     124841
30.0      42279
20.0      20590
60.0       5416
10.0       1780
70.0        119
Name: count, dtype: int64

### Add Rail links and nodes

In [18]:
r_net = metcouncil_roadway.add_rail_links_and_nodes(
    roadway_network = r_net,
    parameters = metcouncil_parameters,
    rail_links_file = os.path.join(rail_dir, 'rail_links.geojson'),
    rail_nodes_file = os.path.join(rail_dir, 'rail_nodes.geojson'),
)

Adding centroid and centroid connector to standard network
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Read 556541 shapes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\shapes.geojson in 27.42.
Finished adding rail links and nodes connectors


In [19]:
# check if missing IDs
# rail nodes does not have osm IDs, they have shst IDs
# rail links does not have osm IDs, they have shst IDs

# if node missing shst id
print(r_net.nodes_df.shst_node_id.isnull().sum())
print(r_net.nodes_df.shst_node_id.nunique())
# if node missing osm id
print(r_net.nodes_df.osm_node_id.isnull().sum() + len(r_net.nodes_df[r_net.nodes_df.osm_node_id==""]))
# if node missing osm id
print(r_net.nodes_df.osm_node_id.dtype)
print(r_net.nodes_df[r_net.nodes_df.osm_node_id == 0])
print(r_net.nodes_df.osm_node_id.nunique())
# if node missing model node id
print(r_net.nodes_df.model_node_id.nunique())

# if link missing 
print(r_net.links_df.shstReferenceId.isnull().sum())
print(r_net.links_df.shstReferenceId.nunique())
print(r_net.links_df.model_link_id.nunique())

# if link missing node id
print(r_net.links_df.fromIntersectionId.isnull().sum())
print(r_net.links_df.toIntersectionId.isnull().sum())
print(r_net.links_df.u.isnull().sum())
print(r_net.links_df[r_net.links_df.u == 0])
print(r_net.links_df.v.isnull().sum())
print(r_net.links_df[r_net.links_df.v == 0])

0
414225
1998
object
Empty GeoDataFrame
Columns: [osm_node_id, shst_node_id, county, drive_access, walk_access, bike_access, model_node_id, rail_only, geometry, X, Y, projects]
Index: []
412228
414225
0
1061652
1061652
0
0
5077
Empty GeoDataFrame
Columns: [shstReferenceId, shape_id, shstGeometryId, fromIntersectionId, toIntersectionId, u, v, nodeIds, wayId, roadClass, oneWay, roundabout, link, oneway, lanes_osm, ref, name, highway, service, width, maxspeed, access, junction, bridge, tunnel, landuse, area, key, forward, backReferenceId, metadata, source, roadway, drive_access, walk_access, bike_access, county, length, A, B, model_link_id, locationReferences, rail_only, geometry, bus_only, distance, projects, managed, price, ML_projects, osm_link_id, centroidconnect, lanes, assign_group, roadway_class]
Index: []

[0 rows x 55 columns]
5077
Empty GeoDataFrame
Columns: [shstReferenceId, shape_id, shstGeometryId, fromIntersectionId, toIntersectionId, u, v, nodeIds, wayId, roadClass, oneWay,

# Create Scenario 00

In [20]:
base_scenario = {"road_net": r_net, "transit_net": transit_net}

In [21]:
version_00_scenario = create_scenario(base_scenario = base_scenario)

Creating Scenario
Base_scenario doesn't contain ['road_net', 'transit_net', 'applied_projects', 'conflicts']
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table st

## create standard file of ver 00
Can’t pickle scenario because of lambda function exists in new NW

In [22]:
write_roadway(version_00_scenario.road_net, file_format="geojson", out_dir= os.path.join(net_dir, 'v00', 'standard_networks'), overwrite=True)
write_transit(version_00_scenario.transit_net, file_format="txt", out_dir= os.path.join(net_dir, 'v00', 'standard_networks'), overwrite=True)

Wrote 6 files to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v00\standard_networks


In [23]:
roadway_net.links_df.model_link_id.max()

1061661

In [24]:
roadway_net.nodes_df.shape

(414225, 12)

In [25]:
roadway_net.nodes_df.model_node_id.max()

417325

In [26]:
roadway_net.links_df.columns

Index(['shstReferenceId', 'shape_id', 'shstGeometryId', 'fromIntersectionId',
       'toIntersectionId', 'u', 'v', 'nodeIds', 'wayId', 'roadClass', 'oneWay',
       'roundabout', 'link', 'oneway', 'lanes_osm', 'ref', 'name', 'highway',
       'service', 'width', 'maxspeed', 'access', 'junction', 'bridge',
       'tunnel', 'landuse', 'area', 'key', 'forward', 'backReferenceId',
       'metadata', 'source', 'roadway', 'drive_access', 'walk_access',
       'bike_access', 'county', 'length', 'A', 'B', 'model_link_id',
       'locationReferences', 'rail_only', 'geometry', 'bus_only', 'distance',
       'projects', 'managed', 'price', 'ML_projects', 'osm_link_id',
       'centroidconnect', 'lanes', 'assign_group', 'roadway_class'],
      dtype='object')

# Create Scenario 00B

In [27]:
#this adds BaseAttribute, walk and bike variables to network, 
#manually calculates external connectors, transit priority, and some roadclass values
version_00b_scenario = create_scenario(
    base_scenario = version_00_scenario,
    project_card_filepath = project_card_dir4
)

Creating Scenario
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Fee

In [28]:
version_00b_scenario.apply_all_projects()

Applying add transit priority part b from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\TransitPriorityB.yml
Selecting using explicit link identifiers.
Final selected links: 434
Applying add transit priority from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\TransitPriority.yml
Selecting using explicit link identifiers.
Final selected links: 794
Applying manual changes to assignment group and roadway class from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\manual_assign_group_roadway_class.wrangler
Applying Project to Roadway Network: manual changes to assignment group and roadway class
cube_wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\cube_wrangler
MetCouncil Wrangler bas

<string>:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

<string>:67: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



Applying add walk and bike attributes from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\add_bike_and_walk_attributes.wrangler
Applying Project to Roadway Network: add walk and bike attributes
Applying year 2015 add centroid connector at external stations from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\external_connectors.yaml


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


# Create Scenario 00C

In [29]:
# this adds BaseCorrections, which correct attributes 
version_00c_scenario = create_scenario(
    base_scenario=version_00b_scenario,
    project_card_filepath = project_card_dir
)

Creating Scenario
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Fee

In [30]:
version_00c_scenario.apply_all_projects()

Applying correct year 2018 assignment group and roadway class from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\year_2018_corrections_assign_group_roadway_class.yml
Selecting using explicit link identifiers.
Final selected links: 1
Applying correct year 2018 assignment group from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\year_2018_corrections_assign_group.yml
Selecting using explicit link identifiers.
Final selected links: 574
Selecting using explicit link identifiers.
Final selected links: 585
Selecting using explicit link identifiers.
Final selected links: 164
Selecting using explicit link identifiers.
Final selected links: 15
Selecting using explicit link identifiers.
Final selected links: 32
Selecting using explicit link identifiers.
Final selected links: 13
Selecting using expl

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Applying nic_split from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\nic_split.yml
Selecting using explicit link identifiers.
Final selected links: 2


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Applying msp_deletes from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\msp_deletes.yml
Selecting using explicit link identifiers.
Final selected links: 72
Applying missing 394 reverse from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\Missing394.yml
Applying network cleanup 1 lanes 7 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\Lanes7.yml
Selecting using explicit link identifiers.
Final selected links: 1
Applying network cleanup 1 lanes 6 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\Lanes6.yml
Selecting using explicit link identifiers.
Final selected links: 2
Applying network 

...

Selecting using explicit link identifiers.
Final selected links: 2


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Applying u of m transitway add links from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp98_UofM.yml
Selecting using explicit link identifiers.
Final selected links: 36
Applying bus routing correction cont2 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp98C.yml


\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\roadway\links\edit.py:329: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


Selecting using explicit link identifiers.
Final selected links: 738
Applying bus routing correction cont from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp98B.yml
Selecting using explicit link identifiers.
Final selected links: 799
Applying bus routing correction from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp98.yml
Selecting using explicit link identifiers.
Final selected links: 799
Applying network cleanup 1 asgngrp 7 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp7.yml
Selecting using explicit link identifiers.
Final selected links: 10
Applying network cleanup 2 asgngrp 6 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Ar

# Create Scenario 00d

In [31]:
# this adds BaseMissingRoads, which add missing roads 
version_00d_scenario = create_scenario(
    base_scenario=version_00c_scenario,
    project_card_filepath = project_card_dir3
)

Creating Scenario
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Fee

In [32]:
version_00d_scenario.apply_all_projects()

Applying wacoutalink from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\wacoutalink.yml
Applying stillwaterblvd from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\stillwaterblvd.yml
Applying section1_columbus from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\section1_columbus.yml


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Applying nicollet2 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\nicollet2.yml
Selecting using explicit link identifiers.
Final selected links: 38


\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\roadway\links\edit.py:329: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


Selecting using explicit link identifiers.
Final selected links: 2
Applying marq_second2 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\marq_second.yml
Applying lex_lake_circlepines2 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\lex_lake_circlepines2.yml


...

Selecting using explicit link identifiers.
Final selected links: 2


\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\roadway\links\edit.py:329: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


Selecting using explicit link identifiers.
Final selected links: 2
Applying 12th_busramps from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\12th_busramps.yml
Selecting using explicit link identifiers.
Final selected links: 2


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Selecting using explicit link identifiers.
Final selected links: 2


# Create Version 01

In [33]:
# this adds BaseCorrections2 , which are newer attribute clean up cards 
# there are too many dependencies to list so I'm running this seperately to make sure it applies last 
version_01_scenario = create_scenario(
    base_scenario=version_00d_scenario,
    project_card_filepath = project_card_dir2
)

Creating Scenario
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Fee

In [34]:
version_01_scenario.apply_all_projects()

Applying clean up 35w ramps from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2_v1\DTRamps.yml
Selecting using explicit link identifiers.
Final selected links: 6
Applying ag2cleanup4 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2_v1\AG2Cleanup4.yml


\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\roadway\links\edit.py:329: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set
\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\roadway\links\edit.py:329: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set
\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrang

Selecting using explicit link identifiers.
Final selected links: 10
Selecting using explicit link identifiers.
Final selected links: 5
Selecting using explicit link identifiers.
Final selected links: 8
Selecting using explicit link identifiers.
Final selected links: 114
Selecting using explicit link identifiers.
Final selected links: 7
Selecting using explicit link identifiers.
Final selected links: 3
Selecting using explicit link identifiers.
Final selected links: 19
Selecting using explicit link identifiers.
Final selected links: 27
Selecting using explicit link identifiers.
Final selected links: 53
Selecting using explicit link identifiers.
Final selected links: 15
Selecting using explicit link identifiers.
Final selected links: 6
Selecting using explicit link identifiers.
Final selected links: 30
Selecting using explicit link identifiers.
Final selected links: 36
Selecting using explicit link identifiers.
Final selected links: 2
Selecting using explicit link identifiers.
Final sele

In [35]:
version_01_scenario.applied_projects

['add transit priority part b',
 'add transit priority',
 'manual changes to assignment group and roadway class',
 'add walk and bike attributes',
 'year 2015 add centroid connector at external stations',
 'correct year 2018 assignment group and roadway class',
 'correct year 2018 assignment group',
 'route2_wash',
 'nic_split',
 'msp_deletes',
 'missing 394 reverse',
 'network cleanup 1 lanes 7',
 'network cleanup 1 lanes 6',
 'network cleanup 2 lanes 5 a',
 'network cleanup 1 lanes 5',
 'network cleanup 2 lanes 4 a',
 'network cleanup 1 lanes 4',
 'network cleanup 2 lanes 3 a',
 'network cleanup 1 lanes 3',
 'network cleanup 2 lanes 2g',
 'network cleanup 2 lanes 2 a',
 'network cleanup 1 lanes 2e',
 'network cleanup 1 lanes 2 d3',
 'network cleanup 1 lanes 2d part 2',
 'network cleanup 1 lanes 2d',
 'network cleanup 1 lanes 2 c3',
 'network cleanup 1 lanes 2c part 2',
 'network cleanup 1 lanes 2c',
 'network cleanup 1 lanes 2b',
 'network cleanup 1 lanes 2a',
 'network cleanup 2 lan

In [36]:
version_01_scenario.road_net.links_df['access'] = version_01_scenario.road_net.links_df['access'].fillna('')
version_01_scenario.road_net.links_df['access'] = version_01_scenario.road_net.links_df['access'].apply(
    lambda x: util.shorten_name(x)
)

In [37]:
version_01_scenario.road_net.links_df['name'] = version_01_scenario.road_net.links_df['name'].fillna('')
version_01_scenario.road_net.links_df['name'] = version_01_scenario.road_net.links_df['name'].apply(
    lambda x: util.shorten_name(x)
)

# Save version 01 standard networks

In [38]:
#i need to do this again once i get all the missing roads added


In [39]:
write_roadway(version_01_scenario.road_net, file_format="geojson", out_dir=os.path.join(net_dir, 'v01', 'standard_networks'), overwrite=True)
write_transit(version_01_scenario.transit_net, file_format="txt", out_dir= os.path.join(net_dir, 'v01', 'standard_networks'), overwrite=True)

Wrote 6 files to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks


## have not updated below this mark - rarely need to export this base
## continue to export
## otherwise continue to notebook 02


# Make Travel Model Network

### Add centroid and centroid connectors

In [40]:
r_net = metcouncil_roadway.add_centroid_and_centroid_connector(
    roadway_network = version_01_scenario.road_net,
    parameters = metcouncil_parameters,
    centroid_file = os.path.join(input_dir, 'standard_networks', 'centroid_node.pickle'),
    centroid_connector_link_file = os.path.join(input_dir, 'standard_networks', 'cc_link.pickle'),
    centroid_connector_shape_file = os.path.join(input_dir, 'standard_networks', 'cc_shape.pickle'),
)

Adding centroid and centroid connector to standard network
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:2025: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt

Finished adding centroid and centroid connectors


### Add Rail access and egress links

In [41]:
r_net = metcouncil_roadway.add_rail_ae_connections(
    r_net,
    metcouncil_parameters
)

cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Creating rail access and egress connection links


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


In [42]:
m_net = metcouncil_roadway.roadway_standard_to_met_council_network(
    r_net,
    metcouncil_parameters    
)

Renaming roadway attributes to be consistent with what metcouncil's model is expecting
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Distance Variable 'distance' already in network. Returning without overwriting.
Finished creating ML lanes variable: ML_lanes
Finished creating hov corridor variable: segment_id
Managed Variable 'managed' already in network. Returning without overwriting.
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Area Type Variable 'area_type' already in network. But some records are missing. Calcualting the missing values without overwriting existing.
Calculating Area Type

\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:275: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_gdf["geometry"] = centroids_gdf["geometry"].centroid


Finished Calculating Area Type from Spatial Data into variable: area_type
Overwriting existing County Variable 'county' already in network
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Adding roadway network variable for county using a spatial join with: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler\metcouncil_data\county\cb_2017_us_county_5m.shp


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:430: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_gdf["geometry"] = centroids_gdf["geometry"].centroid
C:\Users\local_USYS671257\Temp\ipykernel_3092\2851651829.py:1: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  m_net = metcouncil_roadway.roadway_standard_to_met_council_network(
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:434: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of t

Finished Calculating county variable: county
Calculating MPO as roadway network variable: mpo
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Finished calculating MPO variable: mpo
Adding Counts
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Adding Variable AADT using Shared Streets Reference from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler\metcouncil_data\count_mn\mn_count_ShSt_API_match.csv


...

\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:628: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  roadway_net.links_df[x].fillna(0, inplace=True)
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:628: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result

Splitting variables by time period and category


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  roadway_net.nodes_df[x].fillna("", inplace=True)
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:640: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has 

Specified variable to split: ttime_assert not in network variables: Index(['shstReferenceId', 'shape_id', 'shstGeometryId', 'fromIntersectionId',
       'toIntersectionId', 'u', 'v', 'nodeIds', 'wayId', 'roadClass', 'oneWay',
       'roundabout', 'link', 'oneway', 'lanes_osm', 'ref', 'name', 'highway',
       'service', 'width', 'maxspeed', 'access', 'junction', 'bridge',
       'tunnel', 'landuse', 'area', 'key', 'forward', 'backReferenceId',
       'metadata', 'source', 'roadway', 'drive_access', 'walk_access',
       'bike_access', 'county', 'length', 'A', 'B', 'model_link_id',
       'locationReferences', 'rail_only', 'geometry', 'bus_only', 'distance',
       'projects', 'managed', 'price', 'ML_projects', 'osm_link_id',
       'centroidconnect', 'lanes', 'assign_group', 'roadway_class',
       'trn_priority', 'area_type', 'ramp_flag', 'bike', 'walk', 'MNPASS_CODE',
       'MNPASS_PAY', 'segment_id', 'mpo', 'AADT', 'count_AM', 'count_MD',
       'count_PM', 'count_NT', 'count_daily

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


In [43]:
# check if missing IDs
# centroids does not have osm and shst IDs
# centroid connectors does not have osm and shst IDs

# if node missing shst id
print(m_net.nodes_df.shst_node_id.isnull().sum())
print(m_net.nodes_df.shst_node_id.nunique())

# if node missing model node id
print(m_net.nodes_df.model_node_id.nunique())

# if link missing 
print(m_net.links_df.shstReferenceId.isnull().sum() + len(m_net.links_df[m_net.links_df.shstReferenceId==""]))
print(m_net.links_df.shstReferenceId.nunique())
print(m_net.links_df.model_link_id.nunique())

# if link missing node id
print(m_net.links_df.fromIntersectionId.isnull().sum())
print(m_net.links_df.toIntersectionId.isnull().sum())

0
414221
417312
42252
1061504
1103756
21346
21346


In [44]:
m_net.nodes_df.shape

(417312, 13)

In [45]:
m_net.nodes_df.columns

Index(['osm_node_id', 'shst_node_id', 'drive_access', 'walk_access',
       'bike_access', 'model_node_id', 'rail_only', 'geometry', 'X', 'Y',
       'projects', 'county', 'N'],
      dtype='object')

In [46]:
m_net.links_df.columns

Index(['shstReferenceId', 'shape_id', 'shstGeometryId', 'fromIntersectionId',
       'toIntersectionId', 'u', 'v', 'nodeIds', 'wayId', 'roadClass',
       ...
       'price_sov_NT', 'price_hov2_NT', 'price_hov3_NT', 'price_truck_NT',
       'access_EA', 'access_AM', 'access_MD', 'access_PM', 'access_NT',
       'geometry'],
      dtype='object', length=116)

# Write model network as shapefile

In [47]:
#trying to export all of them 
#out_cols = ['model_link_id', 'id', 'assign_group', 'drive_access', 'roadway_class',
#            'lanes_AM', 'lanes_MD', 'lanes_PM', 'lanes_NT', 'segment_id', 'HOV', 'bike', 'walk','roadway',
#            'price_sov_AM', 'geometry', 'managed']

roadway.write_roadway_as_shp(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    output_link_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'links_v01.shp'),
    output_node_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'nodes_v01.shp'),
    #link_output_variables = out_cols,
    data_to_csv = False,
    data_to_dbf = True,
    export_drive_only = False, # if user only wants drive links/nodes in the shapefile
)

Writing Network as Shapefile
Renaming DBF Node Variables
Renaming variables so that they are DBF-safe


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Renaming DBF Link Variables
Renaming variables so that they are DBF-safe
Writing Node Shapes:
 - Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output\fullnet_v01\shapefile\nodes_v01.shp
Writing Link Shapes:
 - Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output\fullnet_v01\shapefile\links_v01.shp


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:834: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  links_dbf_df.to_file(output_link_shp)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'MNPASS_CODE' to 'MNPASS_COD'
  ogr_write(


# Write model network for Cube

In [48]:
roadway.write_roadway_as_fixedwidth(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    zones = metcouncil_parameters.zones,
    output_link_txt = os.path.join(output_network, 'fullnet_v01', 'links.txt'),
    output_node_txt = os.path.join(output_network,  'fullnet_v01','nodes.txt'),
    output_link_header_width_txt = os.path.join(output_network,  'fullnet_v01', 'links_header_width.txt'),
    output_node_header_width_txt = os.path.join(output_network,  'fullnet_v01','nodes_header_width.txt'),
    output_cube_network_script = os.path.join(output_network,  'fullnet_v01',  'make_complete_network_from_fixed_width_file.s'),
)

Starting fixed width conversion
Writing out link database
Writing out link header and width ----
Starting fixed width conversion
Writing out node database
Writing out node header and width


In [49]:
version_01_scenario.transit_net.road_net = version_01_scenario.road_net
standard_transit_net = StandardTransit.fromTransitNetwork(version_01_scenario.transit_net, parameters=metcouncil_parameters)

In [50]:
standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_network, 'fullnet_v01', "line_name_xwalk.csv")
)    

cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Converting GTFS Standard Properties to MetCouncil's Cube Standard
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1253: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1265: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])


In [51]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_network, 'fullnet_v01', "transit.lin"))

In [52]:
version_01_scenario.road_net.nodes_df.model_node_id.max()

896904

In [53]:
version_01_scenario.road_net.nodes_df.model_node_id.shape

(417312,)

In [54]:
m_net.nodes_df.model_node_id.max()

896904

In [55]:
m_net.nodes_df.model_node_id.shape

(417312,)